In [19]:
import pandas as pd
import difflib
import time

# Inicio del cronómetro
inicio = time.time()



########################################################################################################################

#_El programa busca una exactitud del 95% en las coincidencias o +

#_Solo se realiza la busqueda por valores numericos (no string) en la columna CUIT de la pestaña 'Base del mes anterior'

########################################################################################################################



#Abre el archivo de Excel a trabajar
bancos_manuales = pd.read_excel('Contabilidad.xlsx')


# Lee la hoja que contiene la nomina de entidades bancarias "Base del mes anterior"
nomina_entidades = pd.read_excel('Contabilidad.xlsx', sheet_name="Base del mes anterior", skiprows=3)


# Eliminar las filas que tengan valores NaN en la columna 'CUIT'
nomina_entidades = nomina_entidades.dropna(subset=['CUIT'])


# Convertir la columna 'CUIT' a entero, ya que sino queda en formato float64. Ej: '30500001735.0'
lista_entidades = nomina_entidades['CUIT'].astype('int64')


#Luego la convierto a string y posteriormente a una lista
lista_entidades = lista_entidades.astype(str).tolist()


# Convertir los títulos de las columnas a mayúsculas
bancos_manuales.columns = bancos_manuales.columns.str.upper()





def buscar_palabra_con_porcentaje(palabra_objetivo, lista_entidades, porcentaje_minimo=0.95):
    """
    Busca una palabra en una lista y devuelve la coincidencia si la similitud supera un porcentaje mínimo.

    Args:
        palabra_objetivo (str): La palabra a buscar.
        lista_entidades (list): La lista de palabras en la que buscar.
        porcentaje_minimo (float): El porcentaje mínimo de similitud requerido (0 a 1).

    Returns:
        str: La palabra coincidente si se encuentra, o None si no hay coincidencia.
    """
    mejor_coincidencia = None
    mejor_porcentaje = 0

    palabra_objetivo = str(palabra_objetivo)


    for palabra in lista_entidades:
        porcentaje = difflib.SequenceMatcher(None, palabra_objetivo, palabra).ratio()
        if porcentaje > mejor_porcentaje:
            mejor_porcentaje = porcentaje
            mejor_coincidencia = palabra

    if mejor_porcentaje >= porcentaje_minimo:
        return mejor_coincidencia, mejor_porcentaje
    else:
        # Return (None, 0) instead of just None
        return None, 0

    
    
    
        
def aplicar_funcion_dataframe(fila, porcentaje_minimo=0.95):
    """
    Aplica la función buscar_palabra_con_porcentaje a 'Referencia1' y 'Referencia2', y devuelve 'CUIT' o ''.

    Args:
        fila (pd.Series): Una fila del DataFrame.
        porcentaje_minimo (float): El porcentaje mínimo de similitud.

    Returns:
        str: El valor de 'CUIT' o ''.
    """
    
    if pd.isna(fila['CUIT']):  # Verifica si 'CUIT' está vacío
    
    #Primero vamos a buscar una coincidencia del 100%. Una vez la encuentra deja de buscar
        if fila['CLAVE REFERENCIA 1'] in lista_entidades:
            return fila['CLAVE REFERENCIA 1']
        elif fila['CLAVE REFERENCIA 2'] in lista_entidades:
            return fila['CLAVE REFERENCIA 2']
        elif fila['CLAVE REFERENCIA 3'] in lista_entidades:
            return fila['CLAVE REFERENCIA 3']
        else:
    #Sino encontramos una coincidencia del 100%, buscamos el mayor porcentaje de aproximacion        
            coincidencia_referencia1, mejor_porcentaje1 = buscar_palabra_con_porcentaje(fila['CLAVE REFERENCIA 1'], lista_entidades, porcentaje_minimo)


            coincidencia_referencia2, mejor_porcentaje2 = buscar_palabra_con_porcentaje(fila['CLAVE REFERENCIA 2'], lista_entidades, porcentaje_minimo)


            coincidencia_referencia3, mejor_porcentaje3 = buscar_palabra_con_porcentaje(fila['CLAVE REFERENCIA 3'], lista_entidades, porcentaje_minimo)


            #Creamos una lista con los porcentajes, de la fila, en cada una de las columnas
            valores_porcentaje = [mejor_porcentaje1, mejor_porcentaje2, mejor_porcentaje3]


            # Encontrar el porcentaje mas alto de aproximacion
            maximo = max(mejor_porcentaje1, mejor_porcentaje2, mejor_porcentaje3)


            #Encontrar la posición del valor máximo
            #Si hay dos valores máximos en la lista, el método index() de Python siempre devolverá la posición del primer
            #valor máximo que encuentre.
            posicion_maximo = valores_porcentaje.index(maximo)


            #En base a la posicion del maximo de porcentaje obtenemos el valor correspondiente 
            maxima_coincidencia = [coincidencia_referencia1, coincidencia_referencia2, coincidencia_referencia3][int(posicion_maximo)]

            if maxima_coincidencia:
                return maxima_coincidencia

            return ''  # Si no hay coincidencia en ninguna columna, devuelve una cadena vacía

    else:
        return fila['CUIT']  # Conserva el valor original





# Aplica la función a cada fila y los resultados se aplican sobre la columna "CUIT"
bancos_manuales['CUIT'] = bancos_manuales.apply(aplicar_funcion_dataframe, axis=1)


#Creamos un DataFrame que contenga las equivalencias de cada una de las cuit´s
equivalencia_cuit = (nomina_entidades[['CUIT', 'CUIT Relacionada']]).astype('int64')


#Creamos un DF que nos muestre si hay cuit´s duplicadas en la pestaña "Base del mes anterior"
df_duplicados = equivalencia_cuit.loc[equivalencia_cuit.duplicated(), :]


#Copiamos el DF para no modificar el original
duplicados = df_duplicados.copy() 


#Agregamos el texto que las identidfica
duplicados['Base del mes anterior'] = 'cuit duplicada en Base del mes anterior' 


#Borramos del DF 'equivalencia_cuit' las cuit´s duplicadas. Sino al realizar el merge tambien las duplicara
#(crea filas duplicadas en el nuevo DF)
#el metodo .drop_duplicates elimina duplicados basándose en las columnas específicas 'CUIT' y 'CUIT Relacionada'
#ambas columnas deben tener los valores duplicados en forma simultanea
#keep especifica cuál duplicado conservar ('first', 'last', o False para eliminar todos los duplicados)
equivalencia_cuit = equivalencia_cuit.drop_duplicates(subset=['CUIT', 'CUIT Relacionada'], keep='first')


# Convert 'CUIT' column to int64 only for non-NaN values
# using pd.to_numeric with errors='coerce' to handle invalid values
# Replace NaN values with 0 before converting to int64
bancos_manuales['CUIT'] = pd.to_numeric(bancos_manuales['CUIT'], errors='coerce').fillna(0).astype('int64')


#Unimos ambos Dataframes
bancos_manuales = pd.merge(bancos_manuales, equivalencia_cuit, on='CUIT', how='left')


#Reemplazamos los valores de la columna 'CUIT' por los valores de la columna 'CUIT Relacionada'
bancos_manuales['CUIT'] = bancos_manuales['CUIT Relacionada']


#Borramos la columna 'CUIT Relacionada', ya que no es necesaria
bancos_manuales = bancos_manuales.drop(columns=['CUIT Relacionada'])



    ################################
    ########## XLSXWRITER ##########
    ################################


with pd.ExcelWriter("Bancos_manuales.xlsx") as writer:
    bancos_manuales.to_excel(writer, index = False)
    duplicados.to_excel(writer, index = False, startrow = 1, startcol = 20, header=None)

        

# Fin del cronómetro
fin = time.time()

# Tiempo de ejecución
tiempo_ejecucion = fin - inicio
print(f"El tiempo de ejecución es: {tiempo_ejecucion} segundos")

El tiempo de ejecución es: 4.823401689529419 segundos
